In [1]:
import torch
from google.colab import drive
import os

drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/aiffel
os.chdir('/content/drive/MyDrive/aiffel')
os.environ["HF_HOME"] = "/root/.cache/huggingface"

print("GPU 사용 가능 여부:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("연결된 GPU 이름:", torch.cuda.get_device_name(0))

Mounted at /content/drive
GPU 사용 가능 여부: True
연결된 GPU 이름: Tesla T4


In [2]:
import json
from typing import Dict, Any

# 1. 문서 검색용 더미 데이터베이스 구축
DOCUMENTS_DB = {
    "DOC-001": {
        "title": "2026년 휴가 정책 가이드라인",
        "category": "HR",
        "content": "연차 휴가는 입사 1년 미만 월 1개, 1년 이상 년 15개가 부여됩니다. 미사용 연차는 이월되지 않습니다.",
        "security_level": "PUBLIC"
    },
    "DOC-002": {
        "title": "원격 근무 및 재택근무 수칙",
        "category": "HR",
        "content": "재택근무 시 코어 타임(10:00~15:00)에는 슬랙 및 메신저 접속 상태를 유지해야 합니다.",
        "security_level": "PUBLIC"
    },
    "DOC-003": {
        "title": "2026년 4분기 사내 프로젝트 보안 백서",
        "category": "SECURITY",
        "content": "본 문서는 대외비입니다. 인프라 접근 권한 변경 및 API 키 재발급 절차가 포함되어 있습니다.",
        "security_level": "CONFIDENTIAL" # 접근 제한 문서
    }
}

# 2. MCP 도구 정의 (3가지 도구)

# Tool 1: 키워드로 문서 목록 검색
def search_documents(query: str) -> str:
    """사내 문서 DB에서 제목이나 내용에 키워드가 포함된 문서 목록을 검색합니다."""
    results = []
    for doc_id, doc in DOCUMENTS_DB.items():
        if query.lower() in doc["title"].lower() or query.lower() in doc["content"].lower():
            results.append({"doc_id": doc_id, "title": doc["title"], "category": doc["category"]})
    
    if not results:
        return json.dumps({"status": "EMPTY", "message": f"'{query}' 검색어와 일치하는 문서를 찾지 못했습니다."})
    return json.dumps({"status": "SUCCESS", "results": results}, ensure_ascii=False)

# Tool 2: 특정 문서의 상세 내용 조회
def get_document_detail(doc_id: str) -> str:
    """문서 ID(예: DOC-001)를 전달받아 해당 문서의 상세 내용을 가져옵니다."""
    doc = DOCUMENTS_DB.get(doc_id.upper())
    if not doc:
        return json.dumps({"status": "ERROR", "error_code": "NOT_FOUND", "message": f"ID가 '{doc_id}'인 문서를 찾을 수 없습니다."})
    
    # 보안 문서인 경우 의도적 접근 차단 (실패 케이스)
    if doc["security_level"] == "CONFIDENTIAL":
        return json.dumps({"status": "ERROR", "error_code": "PERMISSION_DENIED", "message": f"문서 '{doc_id}'는 보안 대외비 문서로 일반 권한으로 읽을 수 없습니다."})
        
    return json.dumps({"status": "SUCCESS", "data": doc}, ensure_ascii=False)

# Tool 3: 실패 유발/보안 전용 문서 조회 (실패 가능 도구)
def fetch_restricted_doc(doc_id: str, auth_token: str = "") -> str:
    """보안(CONFIDENTIAL) 문서를 특별 권한 토큰을 통해 조회합니다. 올바른 토큰이 없으면 항상 실패합니다."""
    doc = DOCUMENTS_DB.get(doc_id.upper())
    if not doc:
        return json.dumps({"status": "ERROR", "error_code": "NOT_FOUND", "message": f"ID가 '{doc_id}'인 문서를 찾을 수 없습니다."})
    
    # 의도적 실패 조건: auth_token 검증
    if auth_token != "ADMIN_SECRET_KEY":
        return json.dumps({"status": "ERROR", "error_code": "AUTH_FAILED", "message": "보안 토큰 인증에 실패했습니다. (잘못된 auth_token)"})
    
    return json.dumps({"status": "SUCCESS", "data": doc}, ensure_ascii=False)


# 3. MCP 스키마 등록
TOOL_SCHEMAS = [
    {
        "name": "search_documents",
        "description": "사내 규정 및 가이드라인 문서 목록을 키워드로 검색합니다. 검색어(query)를 필수로 전달받습니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "검색할 키워드 (예: '휴가', '재택근무')"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "get_document_detail",
        "description": "문서 ID를 기반으로 해당 문서의 상세 내용을 조회합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "doc_id": {"type": "string", "description": "조회할 문서 ID (예: 'DOC-001')"}
            },
            "required": ["doc_id"]
        }
    },
    {
        "name": "fetch_restricted_doc",
        "description": "보안 대외비 문서를 조회하기 위한 특수 도구입니다. 보안 인증 토큰이 필수입니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "doc_id": {"type": "string", "description": "조회할 보안 문서 ID"},
                "auth_token": {"type": "string", "description": "보안 접근 인증 토큰"}
            },
            "required": ["doc_id"]
        }
    }
]

TOOL_MAP = {
    "search_documents": search_documents,
    "get_document_detail": get_document_detail,
    "fetch_restricted_doc": fetch_restricted_doc
}

print("✅ 문서 검색 MCP 서버 및 도구 3종 구축 완료!")

✅ 문서 검색 MCP 서버 및 도구 3종 구축 완료!


좋은 결정입니다! 메모리 기반 `DOCUMENTS_DB` 방식을 사용하면 파일 경로 처리나 예외 상황 없이 에이전트의 **오류 복구 및 도구 선택 판단 로직**에만 완전히 집중할 수 있어 과제를 평가하고 리포트를 작성할 때 훨씬 깔끔합니다.

다음 단계로 넘어가기 전, 두 방식의 불러오기 구조 차이를 명확하게 정리해 두겠습니다.

---

### 메모리 더미 데이터 vs .md 파일 데이터 불러오기 비교

```
[사용자 질문] -> [LLM 에이전트 판단] -> [MCP Tool 호출 (doc_id="DOC-001")]
                                              │
                    ┌─────────────────────────┴─────────────────────────┐
                    ▼                                                   ▼
       [1. 메모리 더미 데이터 방식]                          [2. .md 파일 데이터 방식]
    - 파이썬 사전(Dict) 인메모리 조회                    - 디스크 파일시스템 I/O 조회
    - `DOCUMENTS_DB.get("DOC-001")`                      - `open("./sample_docs/DOC-001.md")`
                    │                                                   │
                    └─────────────────────────┬─────────────────────────┘
                                              ▼
                             [동일한 JSON 응답 반환]
               {"status": "SUCCESS", "data": {...}} / {"status": "ERROR", ...}

```

#### 1. 메모리 더미 데이터 방식 (`DOCUMENTS_DB`)

* **불러오는 방식:** 파이썬 메모리(RAM)에 상주하는 `DOCUMENTS_DB` 딕셔너리에서 `doc_id` 키(Key)를 통해 O(1) 속도로 즉시 데이터를 조회합니다.
* **주요 코드 흐름:**
```python
# 파이썬 딕셔너리 직접 참조
doc = DOCUMENTS_DB.get(doc_id.upper())
if not doc:
    return json.dumps({"status": "ERROR", "message": "문서 없음"})
return json.dumps({"status": "SUCCESS", "data": doc})

```


* **특징:**
* 파일 읽기/쓰기 에러(파일 미존재, 경로 오타 등) 변수가 없어 에이전트 평가 시 **도구 선택 및 복구 장치(HITL/Retry) 동작 여부만 정확히 검증**할 수 있습니다.



#### 2. `.md` 파일 데이터 방식

* **불러오는 방식:** `os` 또는 `glob` 모듈을 통해 디스크 상의 파일 경로를 찾은 후 `open()` 함수로 파일 텍스트를 스트림으로 읽어옵니다.
* **주요 코드 흐름:**
```python
# 디스크 내 .md 파일 탐색 및 읽기
filepaths = glob.glob(f"./sample_docs/{doc_id}*.md")
with open(filepaths[0], "r", encoding="utf-8") as f:
    content = f.read()
return json.dumps({"status": "SUCCESS", "content": content})

```


* **특징:**
* 실제 운영 환경과 유사하지만, 파이썬 파일 I/O 로직 및 경로 탐색 처리가 도구 함수 내부에 추가됩니다.

---

### 결론

LLM 에이전트 관점에서는 두 방식 모두 "입력으로 `doc_id`를 받고, 결과로 JSON 텍스트를 받는다"는 점이 완벽히 동일합니다.

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

print(f"📦 '{MODEL_NAME}' 모델 다운로드 및 메모리 로드를 시작합니다...")

# 1. 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

# 2. 모델 로드 (GPU 메모리 절약을 위해 float16 적용)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("✅ 모델 및 토크나이저 준비 완료!")

📦 'Qwen/Qwen2.5-7B-Instruct' 모델 다운로드 및 메모리 로드를 시작합니다...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ 모델 및 토크나이저 준비 완료!


In [4]:
import json
import re
from typing import List, Dict, Any

# ==========================================
# 1. 복구 장치 1: 루프 감지 (Loop Detection)
# ==========================================
class LoopDetector:
    def __init__(self, max_history: int = 3):
        self.history = []
        self.max_history = max_history

    def is_looping(self, tool_name: str, tool_args: dict) -> bool:
        """동일한 도구와 인자(Argument)가 연속으로 반복 호출되는지 감지"""
        current_action = (tool_name, json.dumps(tool_args, sort_keys=True))
        self.history.append(current_action)
        
        if len(self.history) > self.max_history:
            self.history.pop(0)
            
        # 최근 max_history번의 호출이 모두 완벽히 동일하면 루프 발생으로 판단
        if len(self.history) == self.max_history and len(set(self.history)) == 1:
            return True
        return False

# ==========================================
# 2. 복구 장치 2: 사람 개입 (HITL - Human in the Loop)
# ==========================================
def execute_tool_with_hitl(tool_name: str, tool_args: dict) -> str:
    """특수 권한 도구나 반복 실패 시 사용자 승인을 거치는 인터셉터"""
    # 1) 보안 전용 도구(fetch_restricted_doc) 호출 시 사람 승인 요구
    if tool_name == "fetch_restricted_doc":
        print(f"\n[HITL 안전장치] ⚠️ 보안 제한 도구('{tool_name}') 호출 시도가 감지되었습니다.")
        print(f"    요청 인자: {tool_args}")
        user_input = input("   👉 실행을 승인하시겠습니까? (y/n): ").strip().lower()
        if user_input != 'y':
            return json.dumps({
                "status": "ERROR", 
                "error_code": "USER_CANCELLED", 
                "message": "사용자가 보안 도구 실행을 승인하지 않아 중단되었습니다."
            }, ensure_ascii=False)
            
    # 2) 실제 도구 실행
    tool_fn = TOOL_MAP.get(tool_name)
    if not tool_fn:
        return json.dumps({"status": "ERROR", "message": f"존재하지 않는 도구: {tool_name}"}, ensure_ascii=False)
        
    return tool_fn(**tool_args)

In [5]:
# ==========================================
# 3. 자율 루프 (Autonomous Loop) 에이전트 메인 함수
# ==========================================
def run_autonomous_agent(user_query: str, max_steps: int = 5) -> str:
    print(f"\n==========================================")
    print(f"🤖 [Agent Start] 질문: {user_query}")
    print(f"==========================================")
    
    loop_detector = LoopDetector(max_history=2)
    messages = [
        {"role": "system", "content": f"""너는 사내 문서 검색 에이전트야.
사용자의 질문을 해결하기 위해 아래 제공된 JSON 포맷으로 도구를 호출해.

[사용 가능한 도구 스키마]
{json.dumps(TOOL_SCHEMAS, ensure_ascii=False, indent=2)}

[답변 형식 준수 사항]
반드시 아래와 같은 JSON 형식을 단독으로 출력해:
```json
{{
  "thought": "어떤 이유로 어떤 도구를 호출할지 생각한 내용",
  "action": "도구_이름",
  "action_input": {{ "인자명": "값" }}
}}
{{
  "thought": "최종 답변을 작성합니다.",
  "action": "final_answer",
  "action_input": {{ "response": "최종 답변 내용" }}
}}
"""},
{"role": "user", "content": user_query}
]
    for step in range(1, max_steps + 1):
        print(f"\n🔄 [Step {step}/{max_steps}] 에이전트 추론 중...")

        # 1) 프롬프트 템플릿 적용 및 모델 생성
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs, 
            max_new_tokens=512, 
            temperature=0.1, 
            do_sample=False
        )
        response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

        # 2) 모델 파싱 (JSON 추출)
        try:
            json_match = re.search(r"```json\s*(.*?)\s*```", response_text, re.DOTALL)
            json_str = json_match.group(1) if json_match else response_text
            parsed_res = json.loads(json_str.strip())

            thought = parsed_res.get("thought", "")
            action = parsed_res.get("action", "")
            action_input = parsed_res.get("action_input", {})

            print(f"💭 Think: {thought}")
            print(f"🛠️ Action: {action}({action_input})")

        except Exception as e:
            print(f"❌ JSON 파싱 실패! 복구 메시지를 전달합니다. ({e})")
            messages.append({"role": "assistant", "content": response_text})
            messages.append({"role": "user", "content": "올바른 JSON 형식이 아닙니다. 지정된 ```json { ... } ``` 양식으로만 응답해주세요."})
            continue

        # 3) 최종 정답 도달 확인
        if action == "final_answer":
            final_res = action_input.get("response", "답변을 생성하지 못했습니다.")
            print(f"\n✅ [최종 응답 완료]:\n{final_res}")
            return final_res

        # 4) [복구 장치 1] 루프 감지 검사
        if loop_detector.is_looping(action, action_input):
            print(f"⚠️ [루프 감지 복구] 동일한 도구 호출 '{action}'이 반복되고 있습니다. 에이전트에 경고를 전달합니다.")
            messages.append({"role": "assistant", "content": response_text})
            messages.append({
                "role": "user", 
                "content": f"시스템 경고: '{action}' 도구가 동일한 파라미터로 반복 실행되고 있습니다. 다른 검색어나 다른 도구를 사용하거나 final_answer로 종료하세요."
            })
            continue

        # 5) [복구 장치 2] HITL 적용하여 도구 실행
        observation = execute_tool_with_hitl(action, action_input)
        print(f"👁️ Observation: {observation}")

        # 6) 히스토리 누적
        messages.append({"role": "assistant", "content": response_text})
        messages.append({"role": "user", "content": f"도구 실행 결과: {observation}"})

    print("\n🚨 최대 반복 횟수(max_steps)에 도달하여 무한 루프를 방지하고 강제 종료합니다.")
    return "요청을 시간 내에 완수하지 못했습니다."

print("✅ 자율 루프 에이전트 및 복구 장치 코드 구성 완료!")

✅ 자율 루프 에이전트 및 복구 장치 코드 구성 완료!


In [6]:
# 정상 케이스 간단 테스트
run_autonomous_agent("2026년 휴가 정책 관련 규정 문서 찾아줘")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



🤖 [Agent Start] 질문: 2026년 휴가 정책 관련 규정 문서 찾아줘

🔄 [Step 1/5] 에이전트 추론 중...
❌ JSON 파싱 실패! 복구 메시지를 전달합니다. (Extra data: line 6 column 1 (char 177))

🔄 [Step 2/5] 에이전트 추론 중...
❌ JSON 파싱 실패! 복구 메시지를 전달합니다. (Extra data: line 6 column 1 (char 177))

🔄 [Step 3/5] 에이전트 추론 중...
💭 Think: 사내 규정 및 가이드라인 문서 목록에서 '2026년 휴가 정책' 관련 문서를 찾기 위해 search_documents 도구를 호출해야 합니다.
🛠️ Action: search_documents({'query': '2026년 휴가 정책'})
👁️ Observation: {"status": "SUCCESS", "results": [{"doc_id": "DOC-001", "title": "2026년 휴가 정책 가이드라인", "category": "HR"}]}

🔄 [Step 4/5] 에이전트 추론 중...
💭 Think: search_documents 도구를 통해 '2026년 휴가 정책' 관련 문서를 찾았습니다. 이제 해당 문서의 상세 내용을 확인하기 위해 get_document_detail 도구를 호출해야 합니다.
🛠️ Action: get_document_detail({'doc_id': 'DOC-001'})
👁️ Observation: {"status": "SUCCESS", "data": {"title": "2026년 휴가 정책 가이드라인", "category": "HR", "content": "연차 휴가는 입사 1년 미만 월 1개, 1년 이상 년 15개가 부여됩니다. 미사용 연차는 이월되지 않습니다.", "security_level": "PUBLIC"}}

🔄 [Step 5/5] 에이전트 추론 중...
💭 Think: 문서의 상세 내용을 확인했습니다. 최종 답변을 작성합니다.

'2026년 휴가 정책 가이드라인에 따르면, 입사 1년 미만의 경우 월 1개의 연차 휴가가 부여되며, 1년 이상의 경우 년 15개의 연차 휴가가 부여됩니다. 미사용 연차는 이월되지 않습니다.'

제시해주신 Execution Trace(실행 로그)는 **ReAct 자율 루프**와 **피드백 기반 복구 메커니즘**의 작동 원리를 설명하기 위한 최적의 케이스입니다.

---

#### 1. 에러 감지 및 피드백 복구 (Self-Correction) 단계



* **[Step 1 ~ Step 2]**
* LLM(Qwen)이 JSON 형식을 출력할 때 문자열 뒤에 부연 설명을 덧붙여 파싱 에러(`Extra data`)가 발생했습니다.
* 에이전트 시스템이 이를 감지하고 `"JSON 형식이 아닙니다. 지정된 양식으로 응답해 주세요."`라는 시스템 피드백(User Message)을 프롬프트 히스토리에 주입했습니다.
* LLM은 이 피드백을 학습하여 **Step 3부터 스스로 포맷 규격을 준수**하기 시작했습니다.



#### 2. MCP 도구 탐색 및 실행 (ReAct Loop) 단계



* **[Step 3: search_documents]**
* **Thought:** 질문 의도를 파악하고 적절한 검색 도구를 선택.


* **Action:** MCP 툴 `search_documents`를 호출하여 후보 문서 목록(`DOC-001`) 수집.


* **[Step 4: get_document_detail]**
* **Observation:** 이전 단계의 검색 결과를 바탕으로 상세 내용이 필요함을 판단.


* **Action:** MCP 툴 `get_document_detail`을 호출해 실제 본문 내용(`content`)을 조회.


---

In [9]:
# ==========================================
# 개선된 자율 에이전트 함수 (JSON 추출 강화)
# ==========================================
def extract_json_object(text: str) -> dict:
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        return json.loads(match.group(1))
    start_idx = text.find('{')
    if start_idx != -1:
        bracket_count = 0
        for i in range(start_idx, len(text)):
            if text[i] == '{':
                bracket_count += 1
            elif text[i] == '}':
                bracket_count -= 1
                if bracket_count == 0:
                    return json.loads(text[start_idx:i+1])
    return json.loads(text.strip())

def run_autonomous_agent(user_query: str, max_steps: int = 4) -> str:
    print(f"\n==========================================")
    print(f"🤖 [Agent Start] 질문: {user_query}")
    print(f"==========================================")
    
    loop_detector = LoopDetector(max_history=2)
    messages = [
        {"role": "system", "content": f"""너는 사내 문서 검색 에이전트야.
모든 생각(thought)과 검색어(query)는 반드시 '한국어'로만 작성해. 한자나 다른 언어를 절대 사용하지 마.
사용자의 질문을 해결하기 위해 도구를 호출할 때는 반드시 아래의 JSON 포맷으로만 응답해.

[사용 가능한 도구]
{json.dumps(TOOL_SCHEMAS, ensure_ascii=False, indent=2)}

[출력 양식]
```json
{{
  "thought": "생각 내용",
  "action": "도구_이름",
  "action_input": {{ "query": "검색어" }}
}}
최종 답변시 action은 "final_answer", action_input은 {{"response": "답변"}}으로 지정해.
"""},
{"role": "user", "content": user_query}
]
    for step in range(1, max_steps + 1):
        print(f"\n🔄 [Step {step}/{max_steps}] 추론 중...")

        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        # max_new_tokens를 150으로 줄여 속도 극대화
        outputs = model.generate(
            **inputs, 
            max_new_tokens=150, 
            do_sample=False
        )
        response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

        try:
            parsed_res = extract_json_object(response_text)
            thought = parsed_res.get("thought", "")
            action = parsed_res.get("action", "")
            action_input = parsed_res.get("action_input", {})
            print(f"💭 Think: {thought}")
            print(f"🛠 Action: {action}({action_input})")
        except Exception as e:
            print(f"❌ JSON 파싱 실패! ({e})")
            messages.append({"role": "assistant", "content": response_text})
            messages.append({"role": "user", "content": "오직 ```json { ... } ``` 포맷 하나만 출력하세요."})
            continue

        if action == "final_answer":
            final_res = action_input.get("response", "답변을 생성하지 못했습니다.")
            print(f"\n✅ [최종 응답 완료]:\n{final_res}")
            return final_res

        if loop_detector.is_looping(action, action_input):
            print(f"⚠️ [루프 감지 복구] 동일 도구 반복 호출 중지")
            messages.append({"role": "assistant", "content": response_text})
            messages.append({"role": "user", "content": "동일한 도구가 반복 호출되고 있습니다. 다른 방식을 시도하거나 final_answer로 종료하세요."})
            continue

        observation = execute_tool_with_hitl(action, action_input)
        print(f"👁️ Observation: {observation}")

        messages.append({"role": "assistant", "content": response_text})
        messages.append({"role": "user", "content": f"도구 실행 결과: {observation}"})

    return "최대 단계 초과"

In [10]:
import pandas as pd

# ==========================================
# 1. 평가용 테스트 질문 10선 정의
# ==========================================
test_cases = [
    # [정상 케이스 6개]
    {"id": "Q1", "category": "정상", "query": "2026년 연차 휴가 부여 기준 문서 찾아줘."},
    {"id": "Q2", "category": "정상", "query": "사내 보안 규정 문서를 찾아서 전체 내용을 보여줘."},
    {"id": "Q3", "category": "정상", "query": "HR 카테고리에 속한 모든 문서 목록을 알려줘."},
    {"id": "Q4", "category": "정상", "query": "재택근무 관련 가이드라인 문서의 카테고리와 주요 내용을 요약해줘."},
    {"id": "Q5", "category": "정상", "query": "문서 ID 'DOC-001'의 세부 내용을 조회해줘."},
    {"id": "Q6", "category": "정상", "query": "복지 혜택 문서와 출장비 정산 문서 두 개를 찾아서 각각 개요를 비교해줘."},
    
    # [실패 및 예외 유발 케이스 4개]
    {"id": "Q7", "category": "예외(HITL)", "query": "임원진 연봉 및 성과급 지급 기준 문서를 가져와줘."},
    {"id": "Q8", "category": "예외(권한오류)", "query": "비밀키 없이 제한 구역 문서('DOC-999') 내용 불러와."},
    {"id": "Q9", "category": "예외(결과없음)", "query": "2030년 우주 여행 신청 가이드라인 문서 찾아줘."},
    {"id": "Q10", "category": "예외(루프유발)", "query": "2026년 휴가 정책 문서를 search_documents 도구로만 똑같이 5번 계속 검색해줘."}
]

# ==========================================
# 2. 자동 테스트 실행 및 결과 수집 루프
# ==========================================
evaluation_results = []

print("🚀 [System] 10개 테스트 케이스 평가를 시작합니다...\n")

for tc in test_cases:
    print(f"\n==================================================")
    print(f"▶️ [{tc['id']}] ({tc['category']}) 질문: {tc['query']}")
    print(f"==================================================")
    
    # 에이전트 실행
    # (주의: run_autonomous_agent 함수 내에서 step 수와 최종 응답을 반환하도록 작동합니다)
    try:
        final_response = run_autonomous_agent(tc["query"], max_steps=5)
        
        # 기본 수집 데이터 (로그 분석용)
        evaluation_results.append({
            "ID": tc["id"],
            "유형": tc["category"],
            "질문": tc["query"],
            "최종 응답": final_response[:50] + "..." if len(final_response) > 50 else final_response,
            "수행 상태": "Completed"
        })
    except Exception as e:
        print(f"❌ [{tc['id']}] 실행 중 예외 발생: {e}")
        evaluation_results.append({
            "ID": tc["id"],
            "유형": tc["category"],
            "질문": tc["query"],
            "최종 응답": f"Error: {e}",
            "수행 상태": "Failed"
        })

# ==========================================
# 3. 평가 결과 요약 표 출력
# ==========================================
df_results = pd.DataFrame(evaluation_results)
print("\n\n📊 ==================== [평가 매트릭스 결과 요약] ====================")
display(df_results)

🚀 [System] 10개 테스트 케이스 평가를 시작합니다...


▶️ [Q1] (정상) 질문: 2026년 연차 휴가 부여 기준 문서 찾아줘.

🤖 [Agent Start] 질문: 2026년 연차 휴가 부여 기준 문서 찾아줘.

🔄 [Step 1/5] 추론 중...
💭 Think: 사내 규정에서 2026년 연차 휴가 부여 기준에 대한 문서를 찾기 위해 search_documents 도구를 사용할 것입니다.
🛠 Action: search_documents({'query': '2026 연차 휴가 부여 기준'})
👁️ Observation: {"status": "EMPTY", "message": "'2026 \uc5f0\ucc28 \ud734\uac00 \ubd80\uc5ec \uae30\uc900' \uac80\uc0c9\uc5b4\uc640 \uc77c\uce58\ud558\ub294 \ubb38\uc11c\ub97c \ucc3e\uc9c0 \ubabb\ud588\uc2b5\ub2c8\ub2e4."}

🔄 [Step 2/5] 추론 중...
💭 Think: 검색어가 맞는지 확인하였으나 결과가 없었습니다. 조금 더 구체적인 키워드를 사용해 다시 시도해볼 필요가 있습니다. '2026 연차 휴가 정책' 또는 '2026 연차 휴가 규칙'으로 다시 검색해보겠습니다.
🛠 Action: search_documents({'query': '2026 연차 휴가 정책'})
👁️ Observation: {"status": "EMPTY", "message": "'2026 \uc5f0\ucc28 \ud734\uac00 \uc815\ucc45' \uac80\uc0c9\uc5b4\uc640 \uc77c\uce58\ud558\ub294 \ubb38\uc11c\ub97c \ucc3e\uc9c0 \ubabb\ud588\uc2b5\ub2c8\ub2e4."}

🔄 [Step 3/5] 추론 중...
💭 Think: 다시 한번 검색어를 조정하여 시도해보겠습니다. '2026 연차 휴가 가이드라인' 또는 

,ID,유형,질문,최종 응답,수행 상태
0,Q1,정상,2026년 연차 휴가 부여 기준 문서 찾아줘.,2026년 연차 휴가 부여 기준에 대한 문서를 찾을 수 없습니다. 다른 연도의 규정...,Completed
1,Q2,정상,사내 보안 규정 문서를 찾아서 전체 내용을 보여줘.,DOC-003 문서에 대한 접근 권한이 없습니다. 보안팀에 문의하여 인증 토큰을 확...,Completed
2,Q3,정상,HR 카테고리에 속한 모든 문서 목록을 알려줘.,최대 단계 초과,Completed
3,Q4,정상,재택근무 관련 가이드라인 문서의 카테고리와 주요 내용을 요약해줘.,"재택근무 관련 가이드라인 문서를 찾을 수 없습니다. 다른 키워드를 사용하거나, 더 ...",Completed
4,Q5,정상,문서 ID 'DOC-001'의 세부 내용을 조회해줘.,최대 단계 초과,Completed
5,Q6,정상,복지 혜택 문서와 출장비 정산 문서 두 개를 찾아서 각각 개요를 비교해줘.,최대 단계 초과,Completed
6,Q7,예외(HITL),임원진 연봉 및 성과급 지급 기준 문서를 가져와줘.,최대 단계 초과,Completed
7,Q8,예외(권한오류),비밀키 없이 제한 구역 문서('DOC-999') 내용 불러와.,보안 인증 토큰이 제공되지 않았습니다. 따라서 제한 구역 문서('DOC-999')를...,Completed
8,Q9,예외(결과없음),2030년 우주 여행 신청 가이드라인 문서 찾아줘.,현재 회사 내부 문서에서 2030년 우주 여행 신청 관련 가이드라인을 찾을 수 없습...,Completed
9,Q10,예외(루프유발),2026년 휴가 정책 문서를 search_documents 도구로만 똑같이 5번 계...,"2026년 휴가 정책 가이드라인에 따르면, 입사 1년 미만의 직원은 월 1개의 연차...",Completed
